# World IK: Summary Table

Этот ноутбук загружает `rollout_data.npz`, считает метрики по каждому типу траектории и показывает таблицу в LaTeX-стиле.

In [1]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
from IPython.display import display, Latex

# Путь к результатам (можно поменять)
eval_dir = Path('/home/maksim/IsaacLab/logs/eval/husky_world_ik_coop/2026-04-09_16-21-29')
npz_path = eval_dir / 'rollout_data.npz'
assert npz_path.exists(), f'Файл не найден: {npz_path}'

label_map = {
    'straight_flat': 'Прямая (фикс. Z)',
    'straight_z': 'Прямая (перем. Z)',
    'sinusoidal_xy': 'Синусоида XY',
    'sinusoidal_full': 'Синусоида XY+Z',
    'high_harmonic': 'Высокая гармоника',
}


In [2]:
data = np.load(npz_path)

traj_errors = {}
for key in data.files:
    m = re.match(r'(.+)_(?:ep\d+|env\d+(?:_seg\d+)?)_error$', key)
    if m is None:
        continue
    traj = m.group(1)
    traj_errors.setdefault(traj, []).append(data[key])

rows = []
for traj, arr_list in sorted(traj_errors.items()):
    all_err = np.concatenate(arr_list)
    rows.append({
        'Тип траектории': label_map.get(traj, traj),
        'Средняя, м': float(np.mean(all_err)),
        'Медиана, м': float(np.median(all_err)),
        'Макс, м': float(np.max(all_err)),
        'СКО, м': float(np.std(all_err)),
        '<5см, %': float(np.mean(all_err < 0.05) * 100.0),
        '<10см, %': float(np.mean(all_err < 0.10) * 100.0),
        'Шагов': int(all_err.shape[0]),
    })

df = pd.DataFrame(rows)
df = df.sort_values('Средняя, м').reset_index(drop=True)
df


,Тип траектории,"Средняя, м","Медиана, м","Макс, м","СКО, м","<5см, %","<10см, %",Шагов
0,Прямая (фикс. Z),0.119268,0.083333,1.409338,0.178576,16.880000,69.093333,3750
1,Прямая (перем. Z),0.157612,0.098847,1.865477,0.230597,11.226667,51.626667,3750
2,Синусоида XY,0.166950,0.090641,1.923019,0.238471,12.933333,58.880000,3750
3,Синусоида XY+Z,0.171628,0.100191,1.900137,0.284392,12.266667,49.840000,3750
4,Высокая гармоника,0.197932,0.101815,2.141132,0.322478,13.013333,49.040000,3750


In [3]:
# Более красивый вывод DataFrame в ноутбуке
styled = (
    df.style
      .format({
          'Средняя, м': '{:.4f}',
          'Медиана, м': '{:.4f}',
          'Макс, м': '{:.4f}',
          'СКО, м': '{:.4f}',
          '<5см, %': '{:.1f}',
          '<10см, %': '{:.1f}',
      })
      .background_gradient(subset=['Средняя, м', 'Медиана, м', 'Макс, м', 'СКО, м'], cmap='YlGn_r')
      .set_caption('World IK: метрики ошибки по типам траекторий')
)
display(styled)


,Тип траектории,"Средняя, м","Медиана, м","Макс, м","СКО, м","<5см, %","<10см, %",Шагов
0,Прямая (фикс. Z),0.1193,0.0833,1.4093,0.1786,16.9,69.1,3750
1,Прямая (перем. Z),0.1576,0.0988,1.8655,0.2306,11.2,51.6,3750
2,Синусоида XY,0.1670,0.0906,1.9230,0.2385,12.9,58.9,3750
3,Синусоида XY+Z,0.1716,0.1002,1.9001,0.2844,12.3,49.8,3750
4,Высокая гармоника,0.1979,0.1018,2.1411,0.3225,13.0,49.0,3750


In [5]:
# LaTeX-представление таблицы (booktabs), удобно для отчета/статьи
latex_df = df.copy()
for c in ['Средняя, м', 'Медиана, м', 'Макс, м', 'СКО, м']:
    latex_df[c] = latex_df[c].map(lambda x: f'{x:.4f}')
for c in ['<5см, %', '<10см, %']:
    latex_df[c] = latex_df[c].map(lambda x: f'{x:.1f}')

latex_table = latex_df.to_latex(index=False, escape=True, caption='World IK: сводка ошибок по типам траекторий', label='tab:worldik_eval', longtable=False)
display(Latex(latex_table))
print(latex_table)


<IPython.core.display.Latex object>

\begin{table}
\caption{World IK: сводка ошибок по типам траекторий}
\label{tab:worldik_eval}
\begin{tabular}{lllllllr}
\toprule
Тип траектории & Средняя, м & Медиана, м & Макс, м & СКО, м & <5см, \% & <10см, \% & Шагов \\
\midrule
Прямая (фикс. Z) & 0.1193 & 0.0833 & 1.4093 & 0.1786 & 16.9 & 69.1 & 3750 \\
Прямая (перем. Z) & 0.1576 & 0.0988 & 1.8655 & 0.2306 & 11.2 & 51.6 & 3750 \\
Синусоида XY & 0.1670 & 0.0906 & 1.9230 & 0.2385 & 12.9 & 58.9 & 3750 \\
Синусоида XY+Z & 0.1716 & 0.1002 & 1.9001 & 0.2844 & 12.3 & 49.8 & 3750 \\
Высокая гармоника & 0.1979 & 0.1018 & 2.1411 & 0.3225 & 13.0 & 49.0 & 3750 \\
\bottomrule
\end{tabular}
\end{table}

